#### A single wide excel file with Sales data across regions is transformed into normalized data, identifying necessary dimensions and facts. 
* The data is read from a file with extension .xls, so the read_excel function is used
* The wide data is modelled into different dimensions and fact
* Each Dimension is represented with a dataframe
* The normalized data is written to a .xlsx file using 'openpyxl' engine
* Each dataframe is written into a separate sheet in the same output excel file
* drop_duplicates, groupby, rank, apply, relativedelta functions have been used in the transformation process.
<br>The original dataset contained data from 2011 to 2014, the dates in the dataset has been modified to between the years 2022 to 2025 using the relativedelta function from dateutil package

In [1]:
import os
import pandas as pd
from dateutil.relativedelta import relativedelta

In [2]:
os.chdir("../Data")
# os.getcwd()
# Read the input files into dataframes

In [3]:
globaldf = pd.read_excel("Global-Superstore.xls", sheet_name='Orders', converters={'Postal Code':str})
# Global-Superstore, Sample - Superstore
# Validate the Stats of the numerical columns in the dataset
globaldf.describe(include="all")
# globaldf["Postal Code"].describe()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,City,State,...,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Cost,Order Priority
count,51290.00000,51290,51290,51290,51290,51290,51290,51290,51290,51290,...,51290,51290,51290,51290,51290.000000,51290.000000,51290.000000,51290.000000,51290.000000,51290
unique,NaN,25035,NaN,NaN,4,1590,795,3,3636,1094,...,10292,3,17,3788,NaN,NaN,NaN,NaN,NaN,4
top,NaN,CA-2014-100111,NaN,NaN,Standard Class,PO-18850,Muhammed Yedwab,Consumer,New York City,California,...,OFF-AR-10003651,Office Supplies,Binders,Staples,NaN,NaN,NaN,NaN,NaN,Medium
freq,NaN,14,NaN,NaN,30775,97,108,26518,915,2001,...,35,31273,6152,227,NaN,NaN,NaN,NaN,NaN,29433
mean,25645.50000,NaN,2013-05-11 21:26:49.155781120,2013-05-15 20:42:42.745174528,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,246.490581,3.476545,0.142908,28.610982,26.375818,NaN
min,1.00000,NaN,2011-01-01 00:00:00,2011-01-03 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.444000,1.000000,0.000000,-6599.978000,0.002000,NaN
25%,12823.25000,NaN,2012-06-19 00:00:00,2012-06-23 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,30.758625,2.000000,0.000000,0.000000,2.610000,NaN
50%,25645.50000,NaN,2013-07-08 00:00:00,2013-07-12 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,85.053000,3.000000,0.000000,9.240000,7.790000,NaN
75%,38467.75000,NaN,2014-05-22 00:00:00,2014-05-26 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,251.053200,5.000000,0.200000,36.810000,24.450000,NaN
max,51290.00000,NaN,2014-12-31 00:00:00,2015-01-07 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,22638.480000,14.000000,0.850000,8399.976000,933.570000,NaN


In [4]:
globaldf.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,City,State,...,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Cost,Order Priority
0,32298,CA-2012-124891,2012-07-31,2012-07-31,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,...,TEC-AC-10003033,Technology,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...,2309.650,7,0.0,762.1845,933.57,Critical
1,26341,IN-2013-77878,2013-02-05,2013-02-07,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,...,FUR-CH-10003950,Furniture,Chairs,"Novimex Executive Leather Armchair, Black",3709.395,9,0.1,-288.7650,923.63,Critical
2,25330,IN-2013-71249,2013-10-17,2013-10-18,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,...,TEC-PH-10004664,Technology,Phones,"Nokia Smart Phone, with Caller ID",5175.171,9,0.1,919.9710,915.49,Medium
3,13524,ES-2013-1579342,2013-01-28,2013-01-30,First Class,KM-16375,Katherine Murray,Home Office,Berlin,Berlin,...,TEC-PH-10004583,Technology,Phones,"Motorola Smart Phone, Cordless",2892.510,5,0.1,-96.5400,910.16,Medium
4,47221,SG-2013-4320,2013-11-05,2013-11-06,Same Day,RH-9495,Rick Hansen,Consumer,Dakar,Dakar,...,TEC-SHA-10000501,Technology,Copiers,"Sharp Wireless Fax, High-Speed",2832.960,8,0.0,311.5200,903.04,Critical


In [5]:
globaldf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Row ID          51290 non-null  int64         
 1   Order ID        51290 non-null  object        
 2   Order Date      51290 non-null  datetime64[ns]
 3   Ship Date       51290 non-null  datetime64[ns]
 4   Ship Mode       51290 non-null  object        
 5   Customer ID     51290 non-null  object        
 6   Customer Name   51290 non-null  object        
 7   Segment         51290 non-null  object        
 8   City            51290 non-null  object        
 9   State           51290 non-null  object        
 10  Country         51290 non-null  object        
 11  Postal Code     9994 non-null   object        
 12  Market          51290 non-null  object        
 13  Region          51290 non-null  object        
 14  Product ID      51290 non-null  object        
 15  Ca

In [6]:
# Customer Dimension
#customerdf = globaldf[['Customer ID', 'Customer Name', 'Segment', 'City', 'State']]
customerdf_tmp = globaldf[['Customer ID', 'Customer Name', 'Segment']]
#customerdf = customerdf.drop_duplicates
customerdf = customerdf_tmp.drop_duplicates(ignore_index = True)
customerdf

,Customer ID,Customer Name,Segment
0,RH-19495,Rick Hansen,Consumer
1,JR-16210,Justin Ritter,Corporate
2,CR-12730,Craig Reiter,Consumer
3,KM-16375,Katherine Murray,Home Office
4,RH-9495,Rick Hansen,Consumer
...,...,...,...
1585,SC-10800,Stuart Calhoun,Consumer
1586,BD-1500,Bradley Drucker,Consumer
1587,RC-9825,Roy Collins,Consumer
1588,MG-7890,Michael Granlund,Home Office


In [7]:
# Ship Mode dimension
shipdf_tmp = globaldf[['Ship Mode']]
shipmdf = shipdf_tmp.drop_duplicates(ignore_index = True)
shipmdf

,Ship Mode
0,Same Day
1,Second Class
2,First Class
3,Standard Class


In [8]:
# Region Dimension
#shipdestdf_tmp = globaldf[['City', 'State', 'Country', 'Market', 'Region']].copy()  
shipdestdf_tmp = globaldf[['City', 'State' , 'Country', 'Market', 'Region']].copy()  
# A copy was used to avoid the error "SettingWithCopyWarning: A value is trying to be set on a copy of a slice from a DataFrame" in subsequent steps
# shipdestdf_tmp['City'] = shipdestdf_tmp.apply(lambda row: 'Encinitas' if row['Postal Code'] == '92024' else row['City'], axis=1)
#shipdestdf_tmp.loc[shipdestdf_tmp['Postal Code'] == '92024', 'City'] = 'Encinitas'

# Records of Cities outside of US had a null Postal Code. So will be using the City and State as the unique identifier for Region
# Two Countries were mapped to multiple Markets and Regions, cleaned it using the conditions as below

shipdestdf_tmp['Market'] = shipdestdf_tmp.apply(lambda row: 'EU' if row['Country'] == 'Austria' else row['Market'], axis=1)
shipdestdf_tmp['Market'] = shipdestdf_tmp.apply(lambda row: 'APAC' if row['Country'] == 'Mongolia' else row['Market'], axis=1)
shipdestdf_tmp['Region'] = shipdestdf_tmp.apply(lambda row: 'Central' if row['Country'] == 'Austria' else row['Region'], axis=1)
shipdestdf_tmp['Region'] = shipdestdf_tmp.apply(lambda row: 'North Asia' if row['Country'] == 'Mongolia' else row['Region'], axis=1)
shipdestdf_tmp['CityState'] = shipdestdf_tmp['City'] + '-' + shipdestdf_tmp['State']
shipdestdf = shipdestdf_tmp.drop_duplicates(ignore_index = True)

#shipdestdf_tmp['CityState'].describe()
#print(shipdestdf.groupby('Postal Code').filter(lambda x: len(x) > 1))
# This gave info on which Postal Code had multiple City values associated with it
#shipdestdf_tmp

#print(shipdestdf.groupby(['City', 'State', 'Country', 'Market']).filter(lambda x: len(x) > 1))
shipdestdf 
# shipdestdf['Postal Code'].astype(str)
# shipdestdf['Postal Code'].isna()

,City,State,Country,Market,Region,CityState
0,New York City,New York,United States,US,East,New York City-New York
1,Wollongong,New South Wales,Australia,APAC,Oceania,Wollongong-New South Wales
2,Brisbane,Queensland,Australia,APAC,Oceania,Brisbane-Queensland
3,Berlin,Berlin,Germany,EU,Central,Berlin-Berlin
4,Dakar,Dakar,Senegal,Africa,Africa,Dakar-Dakar
...,...,...,...,...,...,...
3807,San Luis Obispo,California,United States,US,West,San Luis Obispo-California
3808,Abilene,Texas,United States,US,Central,Abilene-Texas
3809,Felahiye,Kayseri,Turkey,EMEA,EMEA,Felahiye-Kayseri
3810,Lewiston,Idaho,United States,US,West,Lewiston-Idaho


In [9]:
# Product Dimension
producttmp_df = globaldf[['Product ID', 'Product Name', 'Sub-Category', 'Category']].copy()
producttmp_df['ProductRank'] = pd.to_numeric(producttmp_df.groupby('Product ID')['Product Name'].rank(method='first'))

# Many Product IDs had multiple Product Names associated with it, 
# Cleaned the Product dimension so that one Product ID is associated with only one Product Name
productdf = producttmp_df[producttmp_df['ProductRank'] == 1].reset_index(drop=True)
productdf.drop(['ProductRank'], axis=1, inplace=True)
productdf

,Product ID,Product Name,Sub-Category,Category
0,TEC-AC-10003033,Plantronics CS510 - Over-the-Head monaural Wir...,Accessories,Technology
1,FUR-CH-10003950,"Novimex Executive Leather Armchair, Black",Chairs,Furniture
2,TEC-PH-10004664,"Nokia Smart Phone, with Caller ID",Phones,Technology
3,TEC-PH-10004583,"Motorola Smart Phone, Cordless",Phones,Technology
4,TEC-SHA-10000501,"Sharp Wireless Fax, High-Speed",Copiers,Technology
...,...,...,...,...
10287,OFF-FA-10004112,"Stockwell Staples, 12 Pack",Fasteners,Office Supplies
10288,OFF-BI-10003253,"Ibico Index Tab, Economy",Binders,Office Supplies
10289,OFF-BI-10002510,"Acco Index Tab, Clear",Binders,Office Supplies
10290,FUR-ADV-10002329,"Advantus Light Bulb, Erganomic",Furnishings,Furniture


In [10]:
# Returned Orders
returnsdf = pd.read_excel("Global-Superstore.xls", sheet_name='Returns')
# Global-Superstore, Sample - Superstore
# Validate the Stats of the numerical columns in the dataset
# returnsdf.describe(include="all")
returnsdf.drop(['Market'], axis=1, inplace=True)
returnsdf

,Returned,Order ID
0,Yes,MX-2013-168137
1,Yes,US-2011-165316
2,Yes,ES-2013-1525878
3,Yes,CA-2013-118311
4,Yes,ES-2011-1276768
...,...,...
1168,Yes,ES-2013-2639112
1169,Yes,CA-2014-134194
1170,Yes,ES-2012-3246286
1171,Yes,ES-2012-4379168


In [11]:
# Sales Fact, adding years to Order Date and Ship Date to make them current info

globaldf.rename(columns={"Order Date": "Order Date Old", "Ship Date": "Ship Date Old"}, inplace=True)
globaldf['Order Date'] = globaldf['Order Date Old'].apply(lambda x: x + relativedelta(years=11)).dt.date
globaldf['Ship Date'] = globaldf['Ship Date Old'].apply(lambda x: x + relativedelta(years=11)).dt.date
globaldf

#print(type(globaldf))

,Row ID,Order ID,Order Date Old,Ship Date Old,Ship Mode,Customer ID,Customer Name,Segment,City,State,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Cost,Order Priority,Order Date,Ship Date
0,32298,CA-2012-124891,2012-07-31,2012-07-31,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,...,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...,2309.650,7,0.0,762.1845,933.570,Critical,2023-07-31,2023-07-31
1,26341,IN-2013-77878,2013-02-05,2013-02-07,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,...,Chairs,"Novimex Executive Leather Armchair, Black",3709.395,9,0.1,-288.7650,923.630,Critical,2024-02-05,2024-02-07
2,25330,IN-2013-71249,2013-10-17,2013-10-18,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,...,Phones,"Nokia Smart Phone, with Caller ID",5175.171,9,0.1,919.9710,915.490,Medium,2024-10-17,2024-10-18
3,13524,ES-2013-1579342,2013-01-28,2013-01-30,First Class,KM-16375,Katherine Murray,Home Office,Berlin,Berlin,...,Phones,"Motorola Smart Phone, Cordless",2892.510,5,0.1,-96.5400,910.160,Medium,2024-01-28,2024-01-30
4,47221,SG-2013-4320,2013-11-05,2013-11-06,Same Day,RH-9495,Rick Hansen,Consumer,Dakar,Dakar,...,Copiers,"Sharp Wireless Fax, High-Speed",2832.960,8,0.0,311.5200,903.040,Critical,2024-11-05,2024-11-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51285,29002,IN-2014-62366,2014-06-19,2014-06-19,Same Day,KE-16420,Katrina Edelman,Corporate,Kure,Hiroshima,...,Fasteners,"Advantus Thumb Tacks, 12 Pack",65.100,5,0.0,4.5000,0.010,Medium,2025-06-19,2025-06-19
51286,35398,US-2014-102288,2014-06-20,2014-06-24,Standard Class,ZC-21910,Zuschuss Carroll,Consumer,Houston,Texas,...,Appliances,Hoover Replacement Belt for Commercial Guardsm...,0.444,1,0.8,-1.1100,0.010,Medium,2025-06-20,2025-06-24
51287,40470,US-2013-155768,2013-12-02,2013-12-02,Same Day,LB-16795,Laurel Beltran,Home Office,Oxnard,California,...,Envelopes,"#10- 4 1/8"" x 9 1/2"" Security-Tint Envelopes",22.920,3,0.0,11.2308,0.010,High,2024-12-02,2024-12-02
51288,9596,MX-2012-140767,2012-02-18,2012-02-22,Standard Class,RB-19795,Ross Baird,Home Office,Valinhos,São Paulo,...,Binders,"Acco Index Tab, Economy",13.440,2,0.0,2.4000,0.003,Medium,2023-02-18,2023-02-22


In [12]:
globaldf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Row ID          51290 non-null  int64         
 1   Order ID        51290 non-null  object        
 2   Order Date Old  51290 non-null  datetime64[ns]
 3   Ship Date Old   51290 non-null  datetime64[ns]
 4   Ship Mode       51290 non-null  object        
 5   Customer ID     51290 non-null  object        
 6   Customer Name   51290 non-null  object        
 7   Segment         51290 non-null  object        
 8   City            51290 non-null  object        
 9   State           51290 non-null  object        
 10  Country         51290 non-null  object        
 11  Postal Code     9994 non-null   object        
 12  Market          51290 non-null  object        
 13  Region          51290 non-null  object        
 14  Product ID      51290 non-null  object        
 15  Ca

In [13]:
# Writing all the normalized dataframes into an xlsx file on different sheets

with pd.ExcelWriter("Global_Superstore.xlsx", engine='openpyxl', mode='w', date_format="YYYY-MM-DD") as writer:
    customerdf.to_excel(writer, sheet_name='Customer', index=False)
    shipdestdf.to_excel(writer, sheet_name='Region', index=False)
    productdf.to_excel(writer, sheet_name='Product', index=False)
    globaldf[['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'City', 'State', 'Product ID', 
              'Sales', 'Quantity', 'Discount', 'Profit', 'Shipping Cost', 'Order Priority']].to_excel(writer, sheet_name='Sales', index=False)
    returnsdf.to_excel(writer, sheet_name='Returns', index=False)


In [14]:
# if(os.path.exists("Global_Superstore.xlsx") and os.path.isfile("Global_Superstore.xlsx")):
#     #os.remove("Global_Superstore.xlsx")
#     print('This is true')
# else:
#     print('Not true')